# Sepsis Early-Warning — Data Preprocessing

Prepares the PhysioNet/CinC 2019 Sepsis Challenge dataset for modeling: predicting
sepsis onset from hourly ICU vital signs and labs.

**Prerequisite:** run `src/download_data.py` first so `data/sepsis_combined.csv` exists.
This dataset is time-series (multiple hourly rows per patient) — every step below is
careful to operate **within each patient**, never mixing data across patients.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/sepsis_combined.csv")
print(f"Loaded: {df.shape}")
print(f"Unique patients: {df['PatientID'].nunique()}")
df.head()

## 1. Select core features

Most of the 40 raw clinical columns are >90% missing (many labs are only drawn when a
clinician suspects a problem, not on a fixed schedule). We focus on the vitals (nearly
always recorded) and a handful of the most clinically relevant, most-complete labs.

In [ ]:
vitals = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]
key_labs = ["BUN", "Creatinine", "Glucose", "Lactate", "WBC", "Platelets"]

df = df.sort_values(["PatientID", "ICULOS"])
print("Missingness in selected columns:")
print(df[vitals + key_labs].isnull().mean().sort_values(ascending=False))

## 2. Forward-fill within each patient

In clinical time series, a measurement stays clinically relevant until the next one is
taken — e.g. a lactate value from 2 hours ago is still meaningful now. We carry the last
known value forward, **grouped by patient** so no data crosses between patients.

In [ ]:
cols_to_fill = vitals + key_labs
df[cols_to_fill] = df.groupby("PatientID")[cols_to_fill].ffill()
print("Missingness after forward-fill:")
print(df[cols_to_fill].isnull().mean().sort_values(ascending=False))

## 3. Add missingness indicators

Before filling remaining gaps, we record *whether* a lab was ever measured for that
patient-hour. Whether a test was ordered at all can itself be informative — e.g. a
lactate test being ordered signals clinical concern, independent of its value.

In [ ]:
for col in key_labs:
    df[col + "_measured"] = df[col].notna().astype(int)

## 4. Fill remaining gaps with population median

Only gaps *before* a patient's first measurement remain (forward-fill can't fill those).
We use the population median as a neutral default.

In [ ]:
for col in cols_to_fill:
    df[col] = df[col].fillna(df[col].median())

df["HospAdmTime"] = df["HospAdmTime"].fillna(df["HospAdmTime"].median())
print(f"Remaining nulls: {df[cols_to_fill].isnull().sum().sum()}")

## 5. Rolling 6-hour features

A single point-in-time vital sign is less useful than its recent trend. A heart rate of
100 that's been climbing for 6 hours is a very different signal than one that's been
stable at 100. We add rolling mean and std (variability) for the 4 core vitals.

In [ ]:
for col in ["HR", "Resp", "MAP", "Temp"]:
    df[col + "_roll6_mean"] = df.groupby("PatientID")[col].transform(
        lambda s: s.rolling(window=6, min_periods=1).mean()
    )
    df[col + "_roll6_std"] = df.groupby("PatientID")[col].transform(
        lambda s: s.rolling(window=6, min_periods=1).std()
    ).fillna(0)

print("New rolling feature columns added.")
print(f"Shape: {df.shape}")

## 6. Fill static demographics and drop sparse columns

`Age`/`Gender` are static per patient and shouldn't be missing, but we fill defensively.
We drop the remaining columns that are still overwhelmingly missing even after
forward-fill — for a baseline model these add more noise than signal.

In [ ]:
df["Age"] = df.groupby("PatientID")["Age"].transform(lambda s: s.fillna(s.median()))
df["Gender"] = df.groupby("PatientID")["Gender"].transform(lambda s: s.fillna(s.median()))

drop_cols = ["EtCO2", "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST",
             "Alkalinephos", "Calcium", "Chloride", "Bilirubin_direct", "Magnesium",
             "Phosphate", "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb",
             "PTT", "Fibrinogen", "Unit1", "Unit2"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"Final shape: {df.shape}")
print(f"Remaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nSepsisLabel balance:\n{df['SepsisLabel'].value_counts(normalize=True)}")

## 7. Patient-level train/test split

**Critical:** we must split by *patient*, not by row. Splitting by row would put different
hours from the *same* patient's stay into both train and test sets — the model would
partly be tested on data it effectively already saw, inflating performance metrics.

In [ ]:
from sklearn.model_selection import train_test_split

patient_ids = df["PatientID"].unique()
train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)

train_df = df[df["PatientID"].isin(train_ids)]
test_df = df[df["PatientID"].isin(test_ids)]

print(f"Train: {train_df.shape} ({train_df['PatientID'].nunique()} patients)")
print(f"Test:  {test_df.shape} ({test_df['PatientID'].nunique()} patients)")
print(f"Train sepsis rate: {train_df['SepsisLabel'].mean():.4f}")
print(f"Test sepsis rate:  {test_df['SepsisLabel'].mean():.4f}")

## 8. Save processed datasets

In [ ]:
train_df.to_csv("data/sepsis_train.csv", index=False)
test_df.to_csv("data/sepsis_test.csv", index=False)
print("Saved data/sepsis_train.csv and data/sepsis_test.csv")